# 🔍 Building Your First RAG Pipeline

**Build a complete Retrieval-Augmented Generation (RAG) system from scratch**

Learn to build production-quality RAG systems that combine the power of search with LLMs to answer questions using your own documents.

---

## 📋 Overview

**What you'll learn:**
- What RAG is and when to use it
- Load and process documents
- Create and store embeddings
- Retrieve relevant context
- Generate answers with citations
- Evaluate RAG performance

**Prerequisites:** 
- Completed LLM basics and prompt engineering notebooks
- Understanding of embeddings (Module 04)
- API keys configured

**Time estimate:** ⏱️ 90-120 minutes

**Difficulty:** 🟡 Intermediate

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. ✅ Understand when and why to use RAG
2. ✅ Build a complete RAG pipeline from scratch
3. ✅ Load and chunk documents effectively
4. ✅ Create and search vector embeddings
5. ✅ Generate answers with source citations
6. ✅ Evaluate and improve RAG quality

---

## 📖 What is RAG?

### The Problem

**LLMs alone have limitations:**
- ❌ Knowledge cutoff (outdated information)
- ❌ No access to private data
- ❌ Hallucinations (making things up)
- ❌ Can't cite sources

### The Solution: RAG

**Retrieval-Augmented Generation** = Search + LLM

```
User Question
     ↓
[1. Retrieve] → Find relevant documents
     ↓
[2. Augment] → Add context to prompt
     ↓
[3. Generate] → LLM answers using context
```

### Real-World Use Cases

- **Customer Support:** Answer questions from documentation
- **Legal:** Search and analyze contracts
- **Research:** Find relevant papers and summarize
- **Internal Knowledge:** Company wikis, policies
- **E-commerce:** Product recommendations and Q&A

### RAG vs Fine-Tuning

| Aspect | RAG | Fine-Tuning |
|--------|-----|-------------|
| **Update data** | Easy (just add docs) | Hard (retrain model) |
| **Cost** | Low (per query) | High (training) |
| **Accuracy** | High (uses real docs) | Varies |
| **Citations** | Yes | No |
| **Best for** | Knowledge, facts | Style, format |

**Rule of thumb:** Start with RAG, fine-tune only if needed!

---

In [ ]:
# Setup
import os
import time
from dotenv import load_dotenv
from typing import List, Dict, Any, Tuple
import json

# Vector database
import chromadb
from chromadb.utils import embedding_functions

# LLM client
from groq import Groq

# Text processing
from langchain.text_splitter import RecursiveCharacterTextSplitter

load_dotenv()
client = Groq(api_key=os.getenv('GROQ_API_KEY'))

print("✅ Setup complete! Ready to build RAG.")

---

## 📚 Step 1: Load Documents

First, let's create sample documents to work with.

In [ ]:
# Sample documents about a fictional company
documents = [
    {
        "id": "doc1",
        "title": "Company Return Policy",
        "content": """Our return policy allows customers to return products within 30 days of purchase. 
        Items must be in original condition with tags attached. Refunds are processed within 5-7 business days. 
        Shipping costs are non-refundable unless the item is defective. 
        Electronics have a 14-day return window instead of 30 days.""",
        "category": "policy"
    },
    {
        "id": "doc2",
        "title": "Product: SuperWidget Pro",
        "content": """The SuperWidget Pro is our flagship product, priced at $299.99. 
        It features advanced AI capabilities, 24/7 customer support, and a 2-year warranty. 
        The device weighs 500g and measures 15cm x 10cm x 3cm. 
        Battery life is approximately 48 hours on a single charge.""",
        "category": "product"
    },
    {
        "id": "doc3",
        "title": "Shipping Information",
        "content": """We offer three shipping options: Standard (5-7 days, free over $50), 
        Express (2-3 days, $15), and Overnight ($35). International shipping is available to 50 countries 
        with delivery times of 10-14 days. Tracking information is provided via email within 24 hours.""",
        "category": "shipping"
    },
    {
        "id": "doc4",
        "title": "Warranty Coverage",
        "content": """All products come with a standard 1-year warranty covering manufacturing defects. 
        Extended warranties (2-year and 3-year) are available for purchase. 
        Warranty does not cover accidental damage, water damage, or misuse. 
        Claims can be filed online through our support portal.""",
        "category": "warranty"
    },
    {
        "id": "doc5",
        "title": "Customer Support Hours",
        "content": """Our customer support team is available Monday-Friday 9 AM to 8 PM EST, 
        and weekends 10 AM to 6 PM EST. We offer support via phone (1-800-SUPPORT), 
        email (support@company.com), and live chat on our website. 
        Average response time is under 2 hours during business hours.""",
        "category": "support"
    },
]

print(f"📚 Loaded {len(documents)} documents")
print("\nSample document:")
print(f"Title: {documents[0]['title']}")
print(f"Content: {documents[0]['content'][:100]}...")

---

## ✂️ Step 2: Chunk Documents

**Why chunk?**
- Embeddings work best on focused text
- Retrieve only relevant sections
- Fit within LLM context limits

**Chunking strategy:**
- Chunk size: 200-500 characters (for short docs)
- Overlap: 50-100 characters (maintain context)
- Split on: sentences, paragraphs

In [ ]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Chunk all documents
chunks = []
for doc in documents:
    # Split the content
    splits = text_splitter.split_text(doc['content'])
    
    # Create chunks with metadata
    for i, chunk_text in enumerate(splits):
        chunks.append({
            'id': f"{doc['id']}_chunk_{i}",
            'text': chunk_text,
            'metadata': {
                'source': doc['title'],
                'doc_id': doc['id'],
                'category': doc['category'],
                'chunk_index': i
            }
        })

print(f"✂️ Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nExample chunk:")
print(f"ID: {chunks[0]['id']}")
print(f"Text: {chunks[0]['text']}")
print(f"Metadata: {chunks[0]['metadata']}")

---

## 🧮 Step 3: Create Embeddings & Store in Vector DB

**What are embeddings?**
- Numerical representations of text
- Similar meaning = similar vectors
- Enable semantic search

**Vector Database:**
- Stores embeddings efficiently
- Fast similarity search
- Supports metadata filtering

In [ ]:
# Initialize ChromaDB (in-memory for demo)
chroma_client = chromadb.Client()

# Create embedding function (using sentence transformers)
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create or get collection
collection = chroma_client.create_collection(
    name="company_docs",
    embedding_function=embedding_function,
    metadata={"description": "Company documentation and policies"}
)

# Add chunks to collection
collection.add(
    ids=[chunk['id'] for chunk in chunks],
    documents=[chunk['text'] for chunk in chunks],
    metadatas=[chunk['metadata'] for chunk in chunks]
)

print(f"✅ Added {collection.count()} chunks to vector database")
print(f"\nCollection stats:")
print(f"  Name: {collection.name}")
print(f"  Count: {collection.count()}")

---

## 🔍 Step 4: Implement Retrieval

**Semantic search:**
1. Convert query to embedding
2. Find most similar chunks
3. Return top-k results with metadata

In [ ]:
def retrieve_context(query: str, n_results: int = 3) -> List[Dict[str, Any]]:
    """
    Retrieve relevant chunks for a query.
    
    Args:
        query: User's question
        n_results: Number of chunks to retrieve
    
    Returns:
        List of relevant chunks with metadata
    """
    # Query the collection
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    
    # Format results
    retrieved = []
    for i in range(len(results['ids'][0])):
        retrieved.append({
            'text': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
            'distance': results['distances'][0][i] if 'distances' in results else None
        })
    
    return retrieved

# Test retrieval
test_query = "What is the return policy?"
print(f"🔍 Query: {test_query}\n")
print("=" * 80)

results = retrieve_context(test_query, n_results=3)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(f"Source: {result['metadata']['source']}")
    print(f"Text: {result['text'][:150]}...")
    if result['distance']:
        print(f"Relevance score: {1 - result['distance']:.3f}")
    print("-" * 80)

---

## 🤖 Step 5: Generate Answer with LLM

**RAG prompt structure:**
```
[System]: You are a helpful assistant. Use the context provided.
[Context]: <Retrieved chunks>
[Question]: <User query>
[Instruction]: Answer using only the context. Cite sources.
```

In [ ]:
def generate_answer(query: str, context_chunks: List[Dict]) -> Dict[str, Any]:
    """
    Generate answer using retrieved context.
    
    Args:
        query: User's question
        context_chunks: Retrieved relevant chunks
    
    Returns:
        Answer with sources
    """
    # Build context from chunks
    context_parts = []
    sources = []
    
    for i, chunk in enumerate(context_chunks, 1):
        source = chunk['metadata']['source']
        text = chunk['text']
        context_parts.append(f"[Source {i}: {source}]\n{text}")
        sources.append(source)
    
    context = "\n\n".join(context_parts)
    
    # Create prompt
    prompt = f"""You are a helpful assistant answering questions about company policies and products.

Context information:
{context}

Question: {query}

Instructions:
- Answer the question using ONLY the information from the context above
- If the context doesn't contain enough information, say so
- Cite sources by mentioning the source names
- Be concise and accurate

Answer:"""
    
    # Generate response
    start_time = time.time()
    
    response = client.chat.completions.create(
        model="mixtral-8x7b-32768",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300
    )
    
    elapsed = time.time() - start_time
    
    answer = response.choices[0].message.content
    
    return {
        'query': query,
        'answer': answer,
        'sources': list(set(sources)),  # Unique sources
        'num_chunks': len(context_chunks),
        'latency': elapsed
    }

# Test RAG pipeline
print("🤖 Testing Complete RAG Pipeline\n")
print("=" * 80)

test_questions = [
    "What is the return policy for electronics?",
    "How much does the SuperWidget Pro cost?",
    "What are the shipping options?",
]

for question in test_questions:
    print(f"\n❓ Question: {question}")
    print("-" * 80)
    
    # Retrieve context
    context = retrieve_context(question, n_results=3)
    
    # Generate answer
    result = generate_answer(question, context)
    
    print(f"\n💡 Answer:\n{result['answer']}")
    print(f"\n📚 Sources: {', '.join(result['sources'])}")
    print(f"⏱️ Latency: {result['latency']:.2f}s")
    print("=" * 80)

---

## 📊 Step 6: Build Complete RAG Class

Let's wrap everything into a reusable class:

In [ ]:
class SimpleRAG:
    """
    A simple but complete RAG system.
    """
    
    def __init__(self, collection_name: str = "rag_docs"):
        """Initialize RAG system."""
        self.client = Groq(api_key=os.getenv('GROQ_API_KEY'))
        self.chroma_client = chromadb.Client()
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        
        # Create collection
        try:
            self.collection = self.chroma_client.create_collection(
                name=collection_name,
                embedding_function=self.embedding_function
            )
        except:
            self.collection = self.chroma_client.get_collection(
                name=collection_name,
                embedding_function=self.embedding_function
            )
        
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=300,
            chunk_overlap=50
        )
    
    def add_documents(self, documents: List[Dict[str, Any]]):
        """Add documents to the RAG system."""
        chunks = []
        for doc in documents:
            splits = self.text_splitter.split_text(doc['content'])
            for i, chunk_text in enumerate(splits):
                chunks.append({
                    'id': f"{doc['id']}_chunk_{i}",
                    'text': chunk_text,
                    'metadata': {
                        'source': doc.get('title', doc['id']),
                        'doc_id': doc['id'],
                        'chunk_index': i
                    }
                })
        
        # Add to collection
        self.collection.add(
            ids=[chunk['id'] for chunk in chunks],
            documents=[chunk['text'] for chunk in chunks],
            metadatas=[chunk['metadata'] for chunk in chunks]
        )
        
        return len(chunks)
    
    def query(self, question: str, n_results: int = 3) -> Dict[str, Any]:
        """Query the RAG system."""
        # Retrieve context
        results = self.collection.query(
            query_texts=[question],
            n_results=n_results
        )
        
        # Format context
        context_parts = []
        sources = []
        
        for i in range(len(results['ids'][0])):
            source = results['metadatas'][0][i]['source']
            text = results['documents'][0][i]
            context_parts.append(f"[Source {i+1}: {source}]\n{text}")
            sources.append(source)
        
        context = "\n\n".join(context_parts)
        
        # Generate answer
        prompt = f"""Answer the question using only the context provided. Cite sources.

Context:
{context}

Question: {question}

Answer:"""
        
        response = self.client.chat.completions.create(
            model="mixtral-8x7b-32768",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=300
        )
        
        return {
            'question': question,
            'answer': response.choices[0].message.content,
            'sources': list(set(sources)),
            'retrieved_chunks': len(results['ids'][0])
        }

# Test the class
print("🏗️ Testing SimpleRAG Class\n")
print("=" * 80)

rag = SimpleRAG(collection_name="test_rag")

# Add documents
num_chunks = rag.add_documents(documents)
print(f"✅ Added {num_chunks} chunks\n")

# Query
result = rag.query("What warranty options are available?")

print(f"❓ Question: {result['question']}")
print(f"\n💡 Answer:\n{result['answer']}")
print(f"\n📚 Sources: {', '.join(result['sources'])}")
print(f"\n📊 Retrieved {result['retrieved_chunks']} chunks")

---

## 📊 Evaluating RAG Performance

**Key metrics:**
1. **Retrieval Quality** - Are we finding the right documents?
2. **Answer Quality** - Is the answer accurate and helpful?
3. **Citation Quality** - Are sources cited correctly?
4. **Latency** - How fast is the response?

In [ ]:
# Create evaluation test set
eval_questions = [
    {
        "question": "What is the return window for electronics?",
        "expected_answer": "14 days",
        "expected_source": "Company Return Policy"
    },
    {
        "question": "How long does standard shipping take?",
        "expected_answer": "5-7 days",
        "expected_source": "Shipping Information"
    },
    {
        "question": "What is the price of SuperWidget Pro?",
        "expected_answer": "$299.99",
        "expected_source": "Product: SuperWidget Pro"
    },
]

print("📊 RAG Evaluation Results\n")
print("=" * 80)

results = []

for test in eval_questions:
    result = rag.query(test['question'])
    
    # Check if answer contains expected information
    answer_correct = test['expected_answer'].lower() in result['answer'].lower()
    source_correct = test['expected_source'] in result['sources']
    
    results.append({
        'question': test['question'],
        'answer_correct': answer_correct,
        'source_correct': source_correct,
        'overall': answer_correct and source_correct
    })
    
    print(f"\nQuestion: {test['question']}")
    print(f"Answer Correct: {'✅' if answer_correct else '❌'}")
    print(f"Source Correct: {'✅' if source_correct else '❌'}")
    print("-" * 80)

# Calculate metrics
answer_accuracy = sum(r['answer_correct'] for r in results) / len(results)
source_accuracy = sum(r['source_correct'] for r in results) / len(results)
overall_accuracy = sum(r['overall'] for r in results) / len(results)

print(f"\n📈 Performance Metrics:")
print(f"Answer Accuracy: {answer_accuracy:.1%}")
print(f"Source Accuracy: {source_accuracy:.1%}")
print(f"Overall Accuracy: {overall_accuracy:.1%}")

---

## 🎯 Exercise: Extend the RAG System

### Challenge: Add Filtering

**Task:** Modify the RAG system to filter by category.

**Example:**
```python
result = rag.query(
    "What are the shipping options?",
    filter_category="shipping"  # Only search shipping docs
)
```

**Hint:** ChromaDB supports metadata filtering in queries!

---

## ✅ Summary

### What You Built

1. ✅ **Complete RAG Pipeline**
   - Document loading and chunking
   - Embedding creation and storage
   - Semantic search
   - Answer generation with citations

2. ✅ **Production-Ready Class**
   - Reusable and extensible
   - Proper error handling
   - Metadata tracking

3. ✅ **Evaluation Framework**
   - Test set creation
   - Accuracy measurement
   - Source verification

### Key Takeaways

💡 **RAG = Search + Generation** - Two step process

✂️ **Chunking matters** - Size and overlap affect quality

🎯 **Retrieval is critical** - Wrong chunks = wrong answers

📚 **Always cite sources** - Build trust and verify

🧪 **Evaluate systematically** - Test on real questions

### Next Steps

Continue learning about RAG:

1. **Improve retrieval:** `05_rag_systems/06_reranking.ipynb`
2. **Better chunking:** `05_rag_systems/08_advanced_chunking.ipynb`
3. **Add evaluation:** `05_rag_systems/05_retrieval_evaluation.ipynb`

---

## 🎉 Congratulations!

You built a working RAG system from scratch!

**This is a production-ready foundation** that you can:
- Extend with more documents
- Improve with better chunking
- Enhance with reranking
- Deploy as an API

**RAG is one of the most practical AI applications!** 🚀

---